# M2 Colab execution

Frozen MobileNetV3-Large MVP plus checkpoint packaging for the M2 report.

Run the cells in order. The notebook expects the KADID-10k data in the shared
Drive folder (`MyDrive/dip-p28`); if the data or an M2 checkpoint is missing,
the discovery cells say so before any long run starts.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
ROOT = Path("/content/drive/MyDrive/dip-p28")
print("ROOT:", ROOT, "exists:", ROOT.exists())
if ROOT.exists():
    for p in sorted(ROOT.iterdir()):
        size = f"{p.stat().st_size / 1e6:.1f} MB" if p.is_file() else "dir"
        print(f"{p.name:44s} {size}")

In [ ]:
%cd /content
!rm -rf repo
!git clone --depth 1 https://github.com/mo1998/New-No-Reference-Image-Quality-Metric.git repo
%cd repo
!git log --oneline -3

In [ ]:
from pathlib import Path
import json, yaml

ROOT = Path("/content/drive/MyDrive/dip-p28")

known = [
    ROOT / "data/kadid10k/dmos.csv",
    ROOT / "data/kadid10k/kadid10k/dmos.csv",
    ROOT / "kadid10k/dmos.csv",
    ROOT / "data/KADID10K/dmos.csv",
]
dmos = next((p for p in known if p.exists()), None)
if dmos is None:
    dmos = next(iter(sorted(ROOT.glob("**/dmos.csv"))), None)
KADID_ROOT = str(dmos.parent) if dmos else None

known_tid = [
    ROOT / "data/tid2013/mos_with_names.txt",
    ROOT / "data/TID2013/mos_with_names.txt",
    ROOT / "tid2013/mos_with_names.txt",
]
tid = next((p for p in known_tid if p.exists()), None)
if tid is None:
    tid = next(iter(sorted(ROOT.glob("**/mos_with_names.txt"))), None)
TID_ROOT = str(tid.parent) if tid else None

known_koniq = [
    ROOT / "data/koniq/koniq10k_scores_and_distributions.tab",
    ROOT / "data/koniq10k/koniq10k_scores_and_distributions.tab",
]
koniq = next((p for p in known_koniq if p.exists()), None)
if koniq is None:
    koniq = next(iter(sorted(ROOT.glob("**/koniq10k_scores_and_distributions.tab"))), None)
KONIQ_ROOT = str(koniq.parent) if koniq else None

print("KADID_ROOT:", KADID_ROOT)
print("TID_ROOT:  ", TID_ROOT)
print("KONIQ_ROOT:", KONIQ_ROOT)
if KADID_ROOT is None:
    print("KADID zips:", [str(p) for p in sorted(ROOT.glob("**/kadid10k.zip"))][:5])

print("\nRun summaries under ROOT:")
summaries = sorted(ROOT.glob("**/summary.json"))
print("count:", len(summaries))
for p in summaries:
    try:
        s = json.loads(p.read_text())
        extras = (s.get("model_kwargs") or {})
        tid_srocc = (s.get("cross_tid2013") or {}).get("srocc")
        print(f"{p.parent} | model={s.get('model')} | kwargs={extras} | test={s.get('test_srocc')} | tid={tid_srocc}")
    except Exception as exc:
        print(p, "unreadable:", exc)

In [ ]:
%cd /content/repo
import sys, torch
sys.path.insert(0, "src")
from nriqa.config import load_config
from nriqa.models import build_model, count_parameters

cfg = load_config("configs/mobilenetv3_large_frozen_kadid.yaml")
model = build_model(cfg)
print("params:", count_parameters(model))
print("trainable:", count_parameters(model, trainable_only=True))
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("trainable tensors:", len(trainable))
print(trainable[:6])
model.eval()
with torch.no_grad():
    print("output:", model(torch.randn(1, 3, 224, 224)))

In [ ]:
%cd /content/repo
import os, subprocess
assert KADID_ROOT, "KADID-10k not found in Drive"

env = dict(os.environ, PYTHONPATH="src")
cmd = [
    "python", "-m", "nriqa.run",
    "--config", "configs/mobilenetv3_large_frozen_kadid.yaml",
    "--output-dir", str(ROOT / "runs_m2"),
    "--set", f"data.root={KADID_ROOT}",
    "--set", "data.eval_datasets=[kadid10k]",
]
if TID_ROOT:
    cmd += ["--set", f"data.tid_root={TID_ROOT}",
            "--set", "data.eval_datasets=[kadid10k,tid2013]"]
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)

In [ ]:
from pathlib import Path
import hashlib, json, zipfile

runs = sorted((ROOT / "runs_m2").glob("*mobilenetv3_large_frozen*"), key=lambda p: p.stat().st_mtime)
assert runs, "no MobileNetV3-Large run directory found"
run = runs[-1]
print("packaging:", run)

out = ROOT / "published"
out.mkdir(exist_ok=True)
files = ["best.pt", "last.pt", "config.yaml", "environment.json", "summary.json",
         "metrics.jsonl", "predictions_test.csv", "predictions_tid2013.csv"]
keep = [f for f in files if (run / f).exists()]
zpath = out / "m2_mobilenetv3_large_frozen.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in keep:
        zf.write(run / f, arcname=f"{run.name}/{f}")

manifest = {
    "archive": str(zpath),
    "sha256": hashlib.sha256(zpath.read_bytes()).hexdigest(),
    "size_bytes": zpath.stat().st_size,
    "files": keep,
    "summary": json.loads((run / "summary.json").read_text()),
}
(out / "m2_mobilenetv3_large_frozen_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps({k: manifest[k] for k in ["archive", "sha256", "size_bytes", "files"]}, indent=2))

In [ ]:
from pathlib import Path
import json, yaml

ROOT = Path("/content/drive/MyDrive/dip-p28")
candidates = []
for cfg_path in sorted(ROOT.glob("**/config.yaml")):
    try:
        c = yaml.safe_load(cfg_path.read_text())
        model = c.get("model") or {}
        kwargs = model.get("kwargs") or {}
        extra = [str(x).lower() for x in ((c.get("data") or {}).get("extra_datasets") or [])]
        name = str(kwargs.get("model_name", "")).lower()
        if model.get("name") == "timm_regressor" and "siglip" in name and any("koniq" in x for x in extra):
            candidates.append(cfg_path.parent)
    except Exception:
        pass

print("headline candidates:", len(candidates))
for d in candidates:
    summary = d / "summary.json"
    metric = None
    if summary.exists():
        try:
            metric = json.loads(summary.read_text()).get("test_srocc")
        except Exception:
            pass
    print(d, "| test_srocc =", metric)

In [ ]:
# Package the five KonIQ-trained SigLIP-B/16 fold checkpoints. Adjust the
# filter if the discovery cell lists a different folder naming scheme.
from pathlib import Path
import hashlib, json, zipfile

KONIQ_RUNS = [d for d in candidates if "cv65" in str(d).lower() and "koniq" in str(d).lower()]
print("packaging folds:", len(KONIQ_RUNS))
for d in KONIQ_RUNS:
    print(" ", d)

if KONIQ_RUNS:
    out = ROOT / "published"
    out.mkdir(exist_ok=True)
    zpath = out / "m2_headline_siglip_b16_384_koniq.zip"
    manifest = {"folds": []}
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
        for d in KONIQ_RUNS:
            if not (d / "best.pt").exists():
                print("skip (no best.pt):", d)
                continue
            for f in ["best.pt", "config.yaml", "environment.json", "summary.json",
                      "metrics.jsonl", "predictions_test.csv", "predictions_tid2013.csv"]:
                p = d / f
                if p.exists():
                    zf.write(p, arcname=f"{d.name}/{f}")
            manifest["folds"].append({
                "run": str(d),
                "summary": json.loads((d / "summary.json").read_text()),
            })
    manifest["archive"] = str(zpath)
    manifest["sha256"] = hashlib.sha256(zpath.read_bytes()).hexdigest()
    manifest["size_bytes"] = zpath.stat().st_size
    (out / "m2_headline_siglip_b16_384_koniq_manifest.json").write_text(json.dumps(manifest, indent=2))
    print(json.dumps({k: manifest[k] for k in ["archive", "sha256", "size_bytes"]}, indent=2))
    print("folds packed:", len(manifest["folds"]))
else:
    print("No cv65 KonIQ runs found. Check the discovery output before packaging.")

## After the run

1. Download or share `published/m2_mobilenetv3_large_frozen.zip` from Drive.
2. Record the SHA-256 and the run directory in the M2 report and README.
3. If the headline SigLIP folds are present, package them in the last cell and
   add the same archive metadata to the report.